# Exploration des données 


## 1. Imports et paramètres

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
DATA = Path("../data")
SPLIT = DATA / "splits" / "split.csv"
pd.set_option("display.max_columns", 50)
COULEURS = {"functional": "tab:green", "functional needs repair": "tab:orange", "non functional": "tab:red"}

## 2. Chargement

In [ ]:
X = pd.read_csv(DATA / "raw" / "train_values.csv")
y = pd.read_csv(DATA / "raw" / "train_labels.csv")

# Les deux fichiers doivent avoir les mêmes id, dans le même ordre
assert (X["id"] == y["id"]).all()
df = X.merge(y, on="id")
df.shape

## 3. Vingt exemples au hasard

In [ ]:
df.sample(20)

## 4. Types des variables

In [ ]:
num = df.select_dtypes(include="number").columns.drop("id").tolist()
cat = df.columns.difference(num + ["id", "status_group"]).tolist()
print(len(num), "numériques :", num)
print(len(cat), "catégorielles")

## 5. Variable cible

In [ ]:
cible = df["status_group"].value_counts()
print((cible / len(df)).round(3))

cible.plot.bar(color=cible.index.map(COULEURS), title="Répartition des états")
plt.ylabel("nombre de points d'eau")
plt.xticks(rotation=0)
plt.show()

## 6. Valeurs manquantes déclarées

In [ ]:
manq = df.isna().sum()
manq[manq > 0].sort_values(ascending=False)

## 7. Valeurs manquantes cachées : les zéros

In [ ]:
cols_zero = ["amount_tsh", "gps_height", "population", "construction_year", "longitude"]
zeros = (df[cols_zero] == 0).mean().sort_values() * 100

zeros.plot.barh(title="Part de zéros (%)")
plt.show()
zeros.round(1)

In [ ]:
# Longitude = 0 : la latitude vaut alors presque 0, donc le point est hors de Tanzanie
df.loc[df["longitude"] == 0, ["longitude", "latitude"]].describe().loc[["count", "min", "max"]]

## 8. Valeurs manquantes cachées : les textes

In [ ]:
faux = ["0", "-", "unknown", "none"]
textes = {c: df[c].astype(str).str.strip().str.lower().isin(faux).sum() for c in cat}
pd.Series(textes)[lambda s: s > 0].sort_values(ascending=False)

## 9. Nombre de modalités par variable catégorielle

In [ ]:
df[cat].nunique().sort_values(ascending=False)

In [ ]:
# Même modalité écrite de plusieurs façons (majuscules)
print(df["installer"].nunique(), "installateurs bruts")
print(df["installer"].str.lower().str.strip().nunique(), "après passage en minuscules")

## 10. Colonnes constantes ou en double

In [ ]:
print("Valeurs de recorded_by :", df["recorded_by"].unique())

paires = [("quantity", "quantity_group"), ("payment", "payment_type"),
          ("water_quality", "quality_group"), ("source", "source_type"),
          ("waterpoint_type", "waterpoint_type_group"), ("extraction_type", "extraction_type_group")]

# Nombre de modalités de B pour chaque modalité de A : 1 partout = B n'apporte rien de plus
for a, b in paires:
    print(f"{a:16} -> {b:22} max {df.groupby(a)[b].nunique().max()} modalité(s)")

## 11. Liens entre variables et état

In [ ]:
def etat_par(col):
    t = pd.crosstab(df[col], df["status_group"], normalize="index")
    t.plot.barh(stacked=True, color=[COULEURS[c] for c in t.columns], figsize=(8, 3),
                title=f"État selon {col}")
    plt.legend(bbox_to_anchor=(1, 1))
    plt.show()
    return t.round(2)

etat_par("quantity")

In [ ]:
etat_par("waterpoint_type")

In [ ]:
etat_par("payment")

In [ ]:
# Âge du point d'eau au moment du relevé (années de construction connues seulement)
annee = pd.to_datetime(df["date_recorded"]).dt.year
age = (annee - df["construction_year"]).where(df["construction_year"] > 0)

print("Âges négatifs (construit après le relevé) :", (age < 0).sum())

for etat, couleur in COULEURS.items():
    age[df["status_group"] == etat].plot.hist(bins=30, alpha=0.5, density=True, color=couleur, label=etat)
plt.legend()
plt.title("Âge selon l'état")
plt.xlabel("années")
plt.show()
age.groupby(df["status_group"]).median()

In [ ]:
# Carte des points d'eau (coordonnées connues seulement)
geo = df[df["longitude"] > 0]
plt.figure(figsize=(7, 7))
plt.scatter(geo["longitude"], geo["latitude"], s=1, c=geo["status_group"].map(COULEURS))
plt.title("Points d'eau en Tanzanie")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()

## 12. Dates des relevés

In [ ]:
annee.value_counts().sort_index()

## 13. Un même point d'eau peut-il être des deux côtés ?

In [ ]:
variables = df.columns.drop(["id", "status_group"]).tolist()
print("Lignes identiques (hors id) :", df.duplicated(variables, keep=False).sum())

gps = df[df["longitude"] > 0]
print("Points partageant les mêmes coordonnées :", gps.duplicated(["longitude", "latitude"], keep=False).sum())

In [ ]:
# Un groupe = un même lieu (mêmes coordonnées), ou une même ligne si les coordonnées manquent
texte = df[variables].astype(object).where(df[variables].notna(), "NA").astype(str)
ligne = texte.agg("|".join, axis=1)  # toute la ligne en un seul texte
cle_ligne = "ligne" + df.groupby(ligne)["id"].transform("min").astype(str)
cle_gps = df["longitude"].round(6).astype(str) + "/" + df["latitude"].round(6).astype(str)
df["groupe"] = np.where(df["longitude"] > 0, cle_gps, cle_ligne).astype(str)

print(df["groupe"].nunique(), "groupes pour", len(df), "lignes")

## 14. Découpage 70 / 15 / 15, stratifié et par groupe

In [ ]:
# On découpe les groupes (un par ligne), stratifié sur la classe
g = df[["groupe", "status_group"]].drop_duplicates("groupe")
g_train, g_reste = train_test_split(g, test_size=0.30, stratify=g["status_group"], random_state=SEED)
g_val, g_test = train_test_split(g_reste, test_size=0.50, stratify=g_reste["status_group"], random_state=SEED)

# Puis chaque point d'eau prend la part de son groupe
df["part"] = "train"
df.loc[df["groupe"].isin(g_val["groupe"]), "part"] = "val"
df.loc[df["groupe"].isin(g_test["groupe"]), "part"] = "test"
df["part"].value_counts(normalize=True).round(3)

## 15. Vérifications

In [ ]:
# Même proportion de classes dans chaque part
pd.crosstab(df["part"], df["status_group"], normalize="index").round(3)

In [ ]:
# Aucun groupe dans deux parts à la fois
assert (df.groupby("groupe")["part"].nunique() == 1).all()
print("Aucun point d'eau des deux côtés.")

## 16. Enregistrement du découpage figé

In [ ]:
split = df[["id", "part"]].sort_values("id").reset_index(drop=True)

if SPLIT.exists():
    # Le fichier existe déjà : on vérifie seulement qu'on retrouve exactement le même découpage
    assert pd.read_csv(SPLIT).astype(str).equals(split.astype(str)), "Le découpage a changé !"
    print("Découpage identique au fichier figé.")
else:
    split.to_csv(SPLIT, index=False)
    print("Découpage enregistré dans", SPLIT)

## 17. Ce qu'il faut retenir

1. 59 400 points d'eau, 40 variables, trois classes déséquilibrées (7 % à réparer).
2. Des valeurs manquantes cachées : zéros et textes « 0 », « - », « unknown ».
3. Des variables à milliers de modalités et plusieurs colonnes en double.
4. La quantité d'eau est la variable la plus liée à l'état (« dry » : presque toujours hors service).
5. Découpage figé 70 / 15 / 15 dans `data/splits/split.csv`, sans point d'eau des deux côtés.
6. Le test ne sera ouvert qu'une fois, en S5.